# Graded evaluation, run 3: August optimizer settings

Pinned to the adapter whose path contains `augustopt`. Rows and tables
are tagged the same way, so runs 1 and 2 are untouched and all three
adapter datasets can stay attached.

## What came before

| arm_c localize | det `silent_break` anchors | optimizer steps | result |
| --- | --- | --- | --- |
| August | 20 | 105 | **31/32** |
| run 1, mixed conditions | 3 | 66 | 5/32 |
| run 2, single condition | 20 | 66 | 4/32 |
| this run | 20 | 105 | ? |

`icl` was 4/32 in all three, correct on seeds [0, 8, 12, 18] every time.
Check that first: it is the arm that never changed, so if it moves,
something is wrong and the rest of the table cannot be read.

A reminder on two numbers that flattered `arm_c` in earlier runs.
Preservation marked exactly one pair on most replies but rarely the right
one, and the routes were a near-constant 2 or 3 hops against a set where
42 of 64 optima are that length. Both are rechecked near the bottom.

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
%pip install -q -U transformers peft bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 56.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 32.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 94.9 MB/s eta 0:00:00:00:01
Note: you may need to restart the kernel to use updated packages.


## Preflight

In [3]:
import sys, glob, json, time, collections
import torch

def find_dir(marker, root="/kaggle/input"):
    """Directory containing `marker`, at any depth. Kaggle nests datasets
    under /kaggle/input/datasets/<user>/<slug>/ in some workspaces."""
    hits = sorted(glob.glob(os.path.join(root, "**", marker), recursive=True),
                  key=lambda p: (p.count(os.sep), len(p)))
    if not hits:
        raise SystemExit(f"no {marker} under {root}")
    return os.path.dirname(hits[0])

REPO_PATH    = find_dir("resource_mdp.py")
EVAL_PATH    = find_dir("anchors_v22.py")
ADAPTER_MATCH = "augustopt"   # substring picking this run's adapter
_hits = sorted(glob.glob("/kaggle/input/**/adapter_config.json",
                         recursive=True))
_cands = [h for h in _hits if ADAPTER_MATCH in h]
if not _cands and len(_hits) == 1:
    print(f"warning: no path contains {ADAPTER_MATCH!r}, using the only "
          "adapter attached")
    _cands = _hits
if len(_cands) != 1:
    raise SystemExit(
        f"need exactly one adapter matching {ADAPTER_MATCH!r}, found "
        f"{_cands or _hits}. Detach the others or change ADAPTER_MATCH.")
ADAPTER_PATH = os.path.dirname(_cands[0])
PAY_CHANGED  = sorted(glob.glob(
    "/kaggle/input/**/payloads_silent_break_det", recursive=True))[0]
PAY_NOCHANGE = sorted(glob.glob(
    "/kaggle/input/**/payloads_no_change_det", recursive=True))[0]
OUT_DIR = "/kaggle/working"

print("repo:    ", REPO_PATH)
print("eval:    ", EVAL_PATH)
print("adapter: ", ADAPTER_PATH)
print("changed: ", PAY_CHANGED)
print("nochange:", PAY_NOCHANGE)

for f in ("run_pilot.py", "ecpm_parser.py", "explore_agent.py",
          "explore_metrics.py"):
    assert os.path.isfile(os.path.join(REPO_PATH, f)), f"{f} missing from repo"

assert torch.cuda.is_available(), "no GPU: set Accelerator in the sidebar"
GPU = torch.cuda.get_device_name(0)
CAP = torch.cuda.get_device_capability(0)
# is_bf16_supported() counts emulation and returns True on a T4 (7.5),
# which has no native bfloat16. Go by compute capability.
USE_BF16 = CAP[0] >= 8
DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"\ngpu: {GPU} | capability {CAP[0]}.{CAP[1]} | compute dtype: {DTYPE}")

import transformers, peft
print("transformers", transformers.__version__, "| peft", peft.__version__)

repo:     /kaggle/input/datasets/mazwyy/ecpm-repo/ecpm-efe
eval:     /kaggle/input/datasets/mazwyy/ecpm-eval
adapter:  /kaggle/input/datasets/mazwyy/ecpm-anchor-adapter-augustopt/anchor_adapter_augustopt_Qwen2.5-1.5B-Instruct
changed:  /kaggle/input/datasets/mazwyy/ecpm-payloads/payloads_silent_break_det
nochange: /kaggle/input/datasets/mazwyy/ecpm-payloads/payloads_no_change_det

gpu: Tesla T4 | capability 7.5 | compute dtype: torch.float16
transformers 5.17.0 | peft 0.20.0


## Adapter identity

The adapter shipped in the August handoff was a Qwen2.5-**3B** LoRA at
`lora_dropout` 0.05. Loading a 3B adapter on a 1.5B base either errors or
mismatches silently, so check before anything else.

In [4]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

cfg = json.load(open(os.path.join(ADAPTER_PATH, "adapter_config.json")))
assert cfg["base_model_name_or_path"] == MODEL_NAME, (
    f"adapter trained on {cfg['base_model_name_or_path']}, "
    f"this notebook loads {MODEL_NAME}")
assert cfg["lora_dropout"] == 0.0, cfg["lora_dropout"]
print("adapter ok:", cfg["base_model_name_or_path"],
      "| r", cfg["r"], "| alpha", cfg["lora_alpha"])

prov_path = os.path.join(ADAPTER_PATH, "phase1_provenance.json")
if os.path.isfile(prov_path):
    prov = json.load(open(prov_path))
    print("phase 1:", prov["anchor_worlds"], "worlds,",
          prov["anchor_examples"], "examples,",
          prov["optimizer_steps"], "steps, loss",
          round(prov["final_loss"], 4))
else:
    print("no phase1_provenance.json next to the adapter")

adapter ok: Qwen/Qwen2.5-1.5B-Instruct | r 16 | alpha 32
phase 1: 40 worlds, 140 examples, 105 steps, loss 0.1098


## Payloads

`common_seeds` matters. `no_change` constructs on all 80 seeds and the
break family on 65, so an unmatched pairing would compare different
graphs across the two detection conditions.

In [5]:
N_SEEDS = 32          # set to 8 for a quick first pass, then raise
RUN_TAG = "augustopt"   # keeps this run's rows out of run 1's

sys.path.insert(0, EVAL_PATH)
import ecpm_eval as E
E.attach(REPO_PATH)

changed  = E.load_payloads(PAY_CHANGED)
nochange = E.load_payloads(PAY_NOCHANGE)
seeds = E.common_seeds(changed, nochange)[:N_SEEDS]
changed  = [p for p in changed  if p["seed"] in seeds]
nochange = [p for p in nochange if p["seed"] in seeds]
payloads = changed + nochange

n_probes = sum(len(p["probes"]) for p in payloads)
print(f"{len(seeds)} matched seeds: {seeds}")
print(f"{len(payloads)} payloads, {n_probes} probes per arm, "
      f"{n_probes * 3} generations total")
print("distinct start/goal in this set:",
      len({(p['facts']['start'], p['facts']['goal']) for p in changed}))

ecpm_parser attached from /kaggle/input/datasets/mazwyy/ecpm-repo/ecpm-efe (looks like v2.2)
32 matched seeds: [0, 1, 4, 5, 7, 8, 9, 10, 11, 12, 13, 16, 17, 18, 19, 20, 21, 22, 23, 25, 26, 28, 29, 30, 33, 34, 35, 36, 37, 38, 39, 40]
64 payloads, 224 probes per arm, 672 generations total
distinct start/goal in this set: 19


## The ceiling, before any model number

A model at or below `rule_drop` has not beaten counting. On deterministic
`silent_break` the rule is 1.00, so anything here is a capability
threshold rather than a world model. Worth having on screen next to the
model's numbers.

In [6]:
import subprocess
print(subprocess.run([sys.executable,
                      os.path.join(EVAL_PATH, "baselines.py"), PAY_CHANGED],
                     capture_output=True, text=True).stdout)


/kaggle/input/datasets/mazwyy/ecpm-payloads/payloads_silent_break_det  condition=silent_break mode=det n=65
  chance                0.064
  rule_drop             65/65 = 1.00   (node-level 1.00)
  rule_deadB            65/65 = 1.00   (node-level 1.00)
  bayes_flat            65/65 = 1.00   (node-level 1.00)
  bayes_route           57/65 = 0.88   (node-level 0.88)
  target on the estimated period-A route: 57/65
  headroom above the rule: +0.00 (counts only), -0.12 (counts + structure)



## Model

Loaded once. `arm_c` runs with the adapter active, `icl` and `none` run
inside `disable_adapter()`, so the only difference between `icl` and
`arm_c` is phase 1.

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=DTYPE,
                         bnb_4bit_use_double_quant=True)
base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb, device_map={"": 0})
model = PeftModel.from_pretrained(base, ADAPTER_PATH)
model.eval()
model.config.use_cache = True
print("loaded | GPU allocated GB:",
      round(torch.cuda.memory_allocated() / 1e9, 2))

def generate(messages, max_new_tokens):
    enc = tok.apply_chat_template(messages, add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, enc["input_ids"].shape[1]:],
                      skip_special_tokens=True)

def make_ask(use_adapter):
    def ask(messages, max_new_tokens):
        if use_adapter:
            return generate(messages, max_new_tokens)
        with model.disable_adapter():
            return generate(messages, max_new_tokens)
    return ask

ASK = {"none": make_ask(False), "icl": make_ask(False),
       "arm_c": make_ask(True)}

# sanity: the adapter really is doing something
probe_msgs = [{"role": "system", "content": E.SYSTEM},
              {"role": "user", "content": changed[0]["single"]["detection"]}]
print("base :", repr(ASK["icl"](probe_msgs, 32)))
print("armc :", repr(ASK["arm_c"](probe_msgs, 32)))

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

loaded | GPU allocated GB: 1.22
base : '{"changed": false}'
armc : '{"changed": false}'


## Timing probe

One payload through one arm, timed, so you know whether the full run fits
the session before starting it.

In [8]:
t0 = time.time()
_ = E.run_arm([changed[0]], ASK["arm_c"], arm="_timing", mode="single",
              verbose=False)
per_payload = time.time() - t0
total_min = per_payload * len(payloads) * 3 / 60
print(f"{per_payload:.1f}s per payload per arm "
      f"-> about {total_min:.0f} minutes for all three arms")
if total_min > 300:
    print("\nThat will not fit one session. Lower N_SEEDS to 16 and rerun "
          "from the payload cell, or run one arm at a time; the JSONL "
          "resumes either way.")

19.7s per payload per arm -> about 63 minutes for all three arms


## Run

`arm_c` first, then `icl`, then `none`, so a timeout still leaves the two
arms that carry the comparison. Rerunning the cell resumes from the
JSONL.

In [ ]:
ARMS = ["arm_c", "icl", "none"]

rows = []
for arm in ARMS:
    print(f"\n=== {arm} ===")
    t0 = time.time()
    rows += E.run_arm(
        payloads, ASK[arm], arm=arm, mode="single",
        with_evidence=(arm != "none"),
        out_path=os.path.join(OUT_DIR, f"raw_{RUN_TAG}_{arm}.jsonl"))
    print(f"  {arm} done in {(time.time()-t0)/60:.1f} min")
print(f"\n{len(rows)} rows")

## Results

`sens` and `spec` are the detection pair. A constant answerer scores 1.00
on one and 0.00 on the other, which is why a single detection number was
uninterpretable.

`pres_acc` next to `pres_const` is the point: if they match, the arm
answered the majority class. `target_recall` is the informative part
underneath.

In [ ]:
import pandas as pd

COLS = ["arm", "sens", "spec", "localize", "pres_acc", "pres_const",
        "target_recall", "pres_parsed", "route_valid", "route_optimal",
        "mean_regret", "bare_json"]
tab = E.table(rows, arms=ARMS)
display(pd.DataFrame(tab).reindex(columns=COLS))

print("\nroute status by arm")
for r in tab:
    print(" ", r["arm"], r["route_status"])

print("\nMcNemar on localization, exact two-sided")
for a, b in (("arm_c", "icl"), ("icl", "none")):
    print(f"  {a} vs {b}: "
          f"{E.mcnemar([r for r in rows if r['arm']==a], [r for r in rows if r['arm']==b])}")

json.dump(tab, open(os.path.join(OUT_DIR, f"table_{RUN_TAG}.json"), "w"), indent=1)

## Per-seed localization

This is how the old 31/32 was cross-checked against independently
recorded break pairs. `node_only` counts right-node-wrong-action, which
was the shape of the single miss last time.

In [ ]:
loc = {}
for r in rows:
    if r["probe"] == "localization" and r["scored"].get("applicable"):
        p = r["parsed"]
        said = (f"{p.get('node')} {p.get('action')}"
                if p["status"] == "ok" else p["status"])
        loc.setdefault(r["seed"], {})[r["arm"]] = said
        loc[r["seed"]]["gold"] = " ".join(r["target"])
df = pd.DataFrame(loc).T[["gold"] + ARMS]
display(df)

for arm in ARMS:
    exact = sum(df[arm] == df["gold"])
    node = sum(a.split()[0] == g.split()[0]
               for a, g in zip(df[arm], df["gold"]) if " " in a)
    print(f"  {arm:6} exact {exact}/{len(df)}   node-level {node}/{len(df)}")

## Package the outputs

## Are the good numbers real?

Two checks that run-1 failed. Preservation marking exactly one pair is
worthless if it is never the right pair. And a fixed-length route scores
`valid_finite` whenever the optimum happens to be that length.

In [ ]:
# 1. preservation: shape versus content
pres = [r for r in rows if r["arm"] == "arm_c" and r["probe"] == "preservation"]
dist = collections.Counter(
    sum(1 for x in r["parsed"]["pairs"] if x["changed"])
    if r["parsed"]["status"] == "ok" else "bad" for r in pres)
print("arm_c pairs marked changed per reply:", dict(sorted(dist.items(), key=str)))
print("arm_c target_recall (how often the marked pair was the right one):",
      [t["target_recall"] for t in tab if t["arm"] == "arm_c"][0])

# 2. adaptation: is route success a length coincidence
ad = [r for r in rows if r["arm"] == "arm_c" and r["probe"] == "adaptation"]
emitted = collections.Counter(len(r["parsed"]["route"]) for r in ad
                              if r["parsed"]["status"] == "ok")
opt_len = collections.Counter(
    len(p["record"]["oracle"]["post"]["optimal_actions"]) for p in payloads)
print("\narm_c emitted route lengths:", dict(sorted(emitted.items())))
print("optimal route lengths in this set:", dict(sorted(opt_len.items())))
if len(emitted) <= 2:
    print("-> arm_c emits a near-constant route length; any route score is "
          "a coincidence with the optimum, not planning")

# 3. self-consistency: does preservation agree with its own localization
agree = tot = 0
byk = collections.defaultdict(dict)
for r in rows:
    if r["arm"] == "arm_c":
        byk[(r["seed"], r["condition"])][r["probe"]] = r
for k, d in byk.items():
    loc, pr = d.get("localization"), d.get("preservation")
    if not loc or loc["parsed"]["status"] != "ok" or pr["parsed"]["status"] != "ok":
        continue
    said = (loc["parsed"]["node"], loc["parsed"]["action"])
    qs = {(x["node"], x["action"]) for x in pr["parsed"]["pairs"]}
    if said in qs:
        tot += 1
        agree += said in {(x["node"], x["action"])
                          for x in pr["parsed"]["pairs"] if x["changed"]}
print(f"\nlocalization answer was queried on {tot} seeds; preservation "
      f"marked that same pair on {agree}")
print("(August adapter: named the pair then called it unchanged on all 32; "
      "run 1: 2 of 10)")

In [ ]:
import zipfile, shutil

ZIP_PATH = f"{OUT_DIR}/eval_{RUN_TAG}_all.zip"
SKIP = (f"eval_{RUN_TAG}_all.zip", ".ipynb_checkpoints")
n = 0
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for path in sorted(glob.glob(f"{OUT_DIR}/**/*", recursive=True)):
        if not os.path.isfile(path):
            continue
        rel = os.path.relpath(path, OUT_DIR)
        if any(s in rel for s in SKIP):
            continue
        z.write(path, rel)
        n += 1
print(f"{n} files -> {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/1e6:.1f} MB)")
for i in zipfile.ZipFile(ZIP_PATH).namelist():
    print("  ", i)